<a href="https://colab.research.google.com/github/amanjyottkaurr/Spotify-recomedation-system/blob/main/spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("/content/spotify_recommendation_dataset.csv")

In [5]:
print("Shape:", df.shape)

Shape: (1702, 22)


In [6]:
df.head()

,track_id,track_name,artist_name,album_name,genre,release_year,language,danceability,energy,loudness,...,instrumentalness,liveness,valence,tempo,duration_ms,popularity,play_count,user_id,user_rating,mood
0,TRK0151,Ocean Memories,Indigo Pulse,Indigo Diaries,NaN,2013.0,English,0.435949,0.928013,-5.081625,...,0.000930,0.212942,0.755971,122.359250,240529.0,44.0,27.0,U063,4.0,Party
1,TRK0094,Golden Waves,Solar Stories,Solar Stories,NaN,2016.0,Korean,0.781728,0.765414,-6.432487,...,0.220876,0.052840,0.932763,105.482435,210231.0,100.0,45.0,U071,3.0,Energetic
2,TRK0213,Paper City,Nova Heaven,Nova Frequency,Country,2025.0,English,0.596553,0.594694,-8.940630,...,0.013109,0.186902,0.884497,111.886033,267414.0,100.0,52.0,U129,4.0,Romantic
3,TRK0292,Hidden Shadows,Neon District,Neon Frequency,K-Pop,2015.0,Korean,0.674979,0.806930,-6.297647,...,0.021191,0.268684,0.945410,108.298867,140464.0,99.0,43.0,U015,3.0,Energetic
4,TRK0087,Ocean Roads,Electric Lines,Electric Frequency,Classical,2026.0,Hindi,0.376094,0.424635,-14.558685,...,0.899973,0.101962,0.367412,82.644256,NaN,100.0,75.0,U045,5.0,Sad


In [7]:
df.isnull().sum()

,0
track_id,0
track_name,59
artist_name,44
album_name,111
genre,92
release_year,43
language,93
danceability,75
energy,76
loudness,61


In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 27


In [9]:
df = df.drop_duplicates().copy()
print("Duplicates removed!")
print("New shape:", df.shape)

Duplicates removed!
New shape: (1675, 22)


In [10]:
audio_features = ["danceability","energy","loudness","speechiness","acousticness",
    "instrumentalness","liveness","valence","tempo","duration_ms"]
print("Audio features selected!")

Audio features selected!


In [11]:
for column in audio_features:
    df[column] = df[column].fillna(
        df[column].median()
    )
print("Missing audio values handled!")

Missing audio values handled!


In [12]:
songs = df.drop_duplicates(
    subset="track_id").copy()
print("Unique songs:", len(songs))

Unique songs: 395


In [13]:
features = songs[audio_features]
scaler = StandardScaler()
feature_matrix = scaler.fit_transform(features)
print("Content features prepared!")

Content features prepared!


In [14]:
content_similarity = cosine_similarity(
    feature_matrix)
print("Content similarity calculated!")

Content similarity calculated!


In [15]:
ratings = df[ ["user_id", "track_id", "track_name", "user_rating"]].copy()
ratings["user_rating"] = pd.to_numeric(
    ratings["user_rating"],
    errors="coerce")
ratings = ratings.dropna(
    subset=["user_id", "track_id", "user_rating"])
print("Ratings data prepared!")

Ratings data prepared!


In [16]:
user_item = ratings.pivot_table(
    index="user_id",
    columns="track_id",
    values="user_rating",
    aggfunc="mean")
user_item = user_item.fillna(0)
print("User-item matrix created!")
print("Shape:", user_item.shape)

User-item matrix created!
Shape: (150, 393)


In [17]:
collaborative_similarity = cosine_similarity(
    user_item.T)
print("Collaborative similarity calculated!")

Collaborative similarity calculated!


In [18]:
common_tracks = list(set(songs["track_id"]) & set(user_item.columns))
print("Common songs:", len(common_tracks))

Common songs: 393


In [19]:
songs = songs.reset_index(drop=True)
content_index = pd.Series(
    songs.index,index=songs["track_id"])
collaborative_index = pd.Series(
    range(len(user_item.columns)),
    index=user_item.columns)
common_content_indices = [content_index[track]
    for track in common_tracks]
common_collaborative_indices = [
    collaborative_index[track]
    for track in common_tracks]
aligned_content_similarity = content_similarity[
    np.ix_(common_content_indices, common_content_indices)]
aligned_collaborative_similarity = collaborative_similarity[
    np.ix_(common_collaborative_indices, common_collaborative_indices)]
print("Content matrix:", aligned_content_similarity.shape)
print("Collaborative matrix:", aligned_collaborative_similarity.shape)

Content matrix: (393, 393)
Collaborative matrix: (393, 393)


In [20]:
content_index = pd.Series( songs.index,
    index=songs["track_id"])
collaborative_index = pd.Series(
    range(len(user_item.columns)),
    index=user_item.columns)
print("Track mappings created!")

Track mappings created!


In [21]:
def hybrid_recommend(song_id, number=5):
    content_idx = content_index[song_id]
    collaborative_idx = collaborative_index[song_id]
    results = []
    for track_id in common_tracks:
        c_index = content_index[track_id]
        cf_index = collaborative_index[track_id]
        score = (0.5 * content_similarity[content_idx, c_index]
            +
            0.5 * collaborative_similarity[collaborative_idx, cf_index
            ])
        results.append([track_id, score])
    results = pd.DataFrame(results,
        columns=["track_id", "hybrid_score"])
    results = results[results["track_id"] != song_id]
    results = results.sort_values(
      "hybrid_score",ascending=False).head(number)
    return results

In [22]:
track_info = songs[["track_id", "track_name", "artist_name", "genre"]].copy()
track_info.head()

,track_id,track_name,artist_name,genre
0,TRK0151,Ocean Memories,Indigo Pulse,NaN
1,TRK0094,Golden Waves,Solar Stories,NaN
2,TRK0213,Paper City,Nova Heaven,Country
3,TRK0292,Hidden Shadows,Neon District,K-Pop
4,TRK0087,Ocean Roads,Electric Lines,Classical


In [23]:
def show_hybrid_recommendations(song_id, number=5):
    recommendations = hybrid_recommend(song_id,number)
    recommendations = recommendations.merge(
    track_info,on="track_id")
    return recommendations[["track_name","artist_name","genre","hybrid_score"]]

In [24]:
track_info.head(20)

,track_id,track_name,artist_name,genre
0,TRK0151,Ocean Memories,Indigo Pulse,NaN
1,TRK0094,Golden Waves,Solar Stories,NaN
2,TRK0213,Paper City,Nova Heaven,Country
3,TRK0292,Hidden Shadows,Neon District,K-Pop
4,TRK0087,Ocean Roads,Electric Lines,Classical
5,TRK0112,Golden Memories,Nova Heaven,Country
6,TRK0210,Afterglow Stories,Urban Parade,NaN
7,TRK0346,Velvet Paradise,Wild Stories,Jazz
8,TRK0104,Falling Stories,Aurora Heights,Jazz
9,TRK0326,Paper Hearts,Static Motion,Classical


In [25]:
show_hybrid_recommendations("TRK0151", 5)

,track_name,artist_name,genre,hybrid_score
0,Hidden Paradise,Midnight Motion,Rock,0.589675
1,Distant Echoes,Indigo Pulse,Rock,0.580997
2,Golden Dreams,Sapphire Sound,NaN,0.502660
3,Midnight Colours,Silver Hearts,Rock,0.477772
4,Electric Sun,Crimson Garden,Pop,0.468585


In [26]:
track_info = songs[
    ["track_id", "track_name", "artist_name", "genre"]]
track_info.head()

,track_id,track_name,artist_name,genre
0,TRK0151,Ocean Memories,Indigo Pulse,NaN
1,TRK0094,Golden Waves,Solar Stories,NaN
2,TRK0213,Paper City,Nova Heaven,Country
3,TRK0292,Hidden Shadows,Neon District,K-Pop
4,TRK0087,Ocean Roads,Electric Lines,Classical


In [27]:
show_hybrid_recommendations("TRK0151", 5)

,track_name,artist_name,genre,hybrid_score
0,Hidden Paradise,Midnight Motion,Rock,0.589675
1,Distant Echoes,Indigo Pulse,Rock,0.580997
2,Golden Dreams,Sapphire Sound,NaN,0.502660
3,Midnight Colours,Silver Hearts,Rock,0.477772
4,Electric Sun,Crimson Garden,Pop,0.468585


In [28]:
print("Total common songs:", len(common_tracks))
print("\nFirst 20 Track IDs:")
print(common_tracks[:20])

Total common songs: 393

First 20 Track IDs:
['TRK0016', 'TRK0189', 'TRK0010', 'TRK0040', 'TRK0349', 'TRK0097', 'TRK0149', 'TRK0353', 'TRK0286', 'TRK0287', 'TRK0012', 'TRK0251', 'TRK0273', 'TRK0361', 'TRK0311', 'TRK0128', 'TRK0372', 'TRK0050', 'TRK0060', 'TRK0133']


In [29]:
show_hybrid_recommendations(common_tracks[1], 5)

,track_name,artist_name,genre,hybrid_score
0,Silent Motion,Wild Harbor,Electronic,0.527712
1,Distant Signals,Nova Riders,K-Pop,0.475217
2,Velvet Echoes,Silver Sound,K-Pop,0.437852
3,Velvet Stories,Indigo Waves,Pop,0.437642
4,Velvet Fire,Velvet Lines,K-Pop,0.433030


In [30]:
positive_data = ratings[ratings['user_rating'] >= 3].copy()
user_counts = positive_data["user_id"].value_counts()
active_users = user_counts[
    user_counts >= 3
].index
evaluation_data = positive_data[
    positive_data["user_id"].isin(active_users)
].copy()

# Redefine test_data to sample from evaluation_data to ensure index compatibility
test_data = evaluation_data.groupby(
    "user_id"
).sample(
    n=1,random_state=42)
print("Test samples:", len(test_data))

train_data = evaluation_data.drop(
    test_data.index)
print("Training interactions:", len(train_data))

Test samples: 150
Training interactions: 1362


In [31]:
test_data = ratings.groupby(
    "user_id"
).sample(
    n=1,random_state=42)
print("Test samples:", len(test_data))

Test samples: 150
